# 과제 1 — 잘못된 SQL 고치기 (20점)

- **범위 · 제출**: 7 · 8주차 범위 · 노트북 파일 하나(`과제1.ipynb`) · 9주차 수업 후 1주일 이내 LMS 제출 (슬라이드 258)
- **이 과제에서 확인하는 것**
  1. 오류 메시지를 읽고 SQL 의 실수 한 군데를 찾아 고칠 수 있다
  2. 무엇이 왜 틀렸는지 한 줄로 설명할 수 있다
  3. 배정 질환의 결과 숫자를 문장으로 해석하고 검산할 수 있다
- **하는 일**: 동료가 쓴 SQL 12개에 **실수가 한 군데씩** 있어 그대로 실행하면 오류가 납니다. 오류 메시지를 읽고 한 군데를 고쳐 ✅ 를 받은 뒤, 무엇이 왜 틀렸는지 한 줄로 적습니다 (1~3절). 마지막에 내 숫자로 해석을 씁니다 (4절).
- **배정 질환**: 학번에 따라 질환이 다르게 배정됩니다. Q10~Q12 는 SQL 의 `'{{MY_MESH}}'` 자리에 내 질환이 자동으로 들어가므로 질환 이름을 적지 않습니다. 결과 숫자는 사람마다 다릅니다.

| 채점 | 점수 | 내용 |
|---|---|---|
| 자동 확인 | 12점 | ① 조회 기초 Q1~Q6 · ② 집계와 GROUP BY Q7~Q9 · ③ 배정 질환 JOIN Q10~Q12. `check` 가 ✅ 이면 1점씩 |
| 결과 해석 | 5점 | 해석1 (3점) · 해석2 (2점). 내 Q10~Q12 숫자로 문장 쓰기 |
| 실행 완료 · 형식 | 3점 | 모든 셀 실행 · 학번과 이름 · **문항마다 ‘무엇이 왜 틀렸나’ 한 줄 (12개)** |

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.
- ✏️ 로 시작하는 칸의 첫 줄은 채점 이름표입니다. 지우지 말고 그 아래 줄에 적으세요.

### 고치는 방법 (12문항 모두 같습니다)

1. 동료의 SQL 셀을 **그대로 실행**해 오류 메시지를 봅니다.
2. **실수 한 군데**를 고쳐 다시 실행합니다. `AS` 로 붙인 결과 열 이름과 `'{{MY_MESH}}'` 는 그대로 둡니다.
3. 바로 아래 `check("Q1", _)` 셀을 실행합니다. ✅ 가 나오면 1점입니다.
4. 그 아래 ✏️ 칸을 두 번 눌러 `→` 뒤에 **무엇이 왜 틀렸나** 를 한 줄로 적습니다 (형식 점수에 들어갑니다).
   예) `→ brief_titel 을 brief_title 로 고침 — 열 이름 철자가 틀려 열을 찾지 못했다`
5. 못 고친 문항은 두고 다음으로 넘어가도 됩니다. `런타임 → 모두 실행` 이 그 셀에서 멈추면, 그 아래 셀을 누르고 `런타임 → 이후 셀 실행` 을 누르세요.

**오류 메시지 읽는 법** (슬라이드 193) — 붉은 글씨 중 **`Error:` 가 들어간 줄**이 진짜 원인입니다 (예: `Binder Error: …`, `Parser Error: …`).

| 메시지에 이 말이 있으면 | 뜻 |
|---|---|
| `Referenced column "…" not found` | 열 이름 오타 또는 따옴표 누락 |
| `Parser Error: syntax error at or near "…"` | 절의 순서나 쉼표 문제 |
| `Conversion Error` / `Cannot compare` | 타입이 맞지 않음 (5주차) |

막히면 (슬라이드 259): `check` 가 '오류가 났습니다' 이면 `Error:` 가 들어간 줄을 읽고 한 군데만 고치세요. 결과가 0 이면 문자 값의 대소문자와 따옴표를 확인하세요. 숫자가 친구와 다른 것은 배정 질환이 달라서이므로 정상입니다.

In [ ]:
#@title ✏️ 학번과 이름을 입력한 뒤 이 셀을 실행하세요 { display-mode: "form" }
학번 = ""  #@param {type:"string"}
이름 = ""  #@param {type:"string"}

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게
%sql conn --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    _LAST.update(tag=tag, error=not result.success)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 학번이 입력되지 않아 배정 질환이 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

MESH = ["Lung Neoplasms", "Parkinson Disease", "Atrial Fibrillation", "Breast Neoplasms", "Depressive Disorder", "Coronary Artery Disease", "Heart Failure", "Pulmonary Disease, Chronic Obstructive", "Renal Insufficiency, Chronic", "Alzheimer Disease", "Obesity", "COVID-19", "Osteoarthritis", "Sepsis", "Pneumonia"]
학번, 이름 = str(globals().get("학번", "")).strip(), str(globals().get("이름", "")).strip()
if not 학번 or 학번 == "20260000":
    MY_MESH = None
    print("⚠️ 학번이 비어 있습니다. 맨 위 셀에 학번과 이름을 입력하고, 그 셀과 이 준비 셀을 다시 실행하세요.")
else:
    MY_MESH = MESH[int(_hl.md5(학번.encode()).hexdigest(), 16) % len(MESH)]
    print(f"연결 완료.  {이름}({학번}) 님의 배정 질환:  {MY_MESH}")
    print("→ SQL 의 '{{MY_MESH}}' 자리에 이 질환이 자동으로 들어갑니다. 질환 이름을 직접 적지 않아도 됩니다.")

def 제출확인():
    """제출 전에 실행: 문항별 결과와 제출 방법을 보여 줍니다."""
    print(f"제출자: {이름 or '(이름 없음)'} ({학번 or '학번 없음'})  /  배정 질환: {MY_MESH}")
    진행확인()
    print("\n제출 방법: ① 메뉴 런타임 → 모두 실행 (못 고친 문항에서 멈추면 그 아래 셀을 누르고 런타임 → 이후 셀 실행)")
    print("           ② 위 결과 확인  ③ 파일 → 다운로드 → .ipynb 다운로드")
    print(f"           ④ 파일 이름을 과제1_{학번 or '학번'}_{이름 or '이름'}.ipynb 로 바꿔 LMS 에 제출")

In [ ]:
#@title 채점용 정보 (수정하지 마세요) { display-mode: "form" }
ANSWERS = {'Q1': {'sig': '8540c99cb945', 'osig': '8540c99cb945', 'rows': 1, 'cols': ['모집중'], 'hint': 'RECRUITING 은 열이 아니라 값입니다. 오류에 나온 후보 열 이름으로 바꾸지 말고, 값의 철자 · 대소문자는 그대로 둡니다 (슬라이드 179).', 'ordered': False, 'optional': False}, 'Q2': {'sig': 'e9b73be38d9d', 'osig': 'e9b73be38d9d', 'rows': 1, 'cols': ['시험수'], 'hint': "'그날 포함' 은 >= 입니다. 날짜는 연-월-일 모양 그대로 작은따옴표 안에 적습니다 (슬라이드 179).", 'ordered': False, 'optional': False}, 'Q3': {'sig': '27fb2750de33', 'osig': '27fb2750de33', 'rows': 1, 'cols': ['시험수'], 'hint': "'이상' 은 >= 입니다. 숫자는 따옴표 없이 숫자만 씁니다 (슬라이드 179).", 'ordered': False, 'optional': False}, 'Q4': {'sig': '0df03d2907f5', 'osig': '0df03d2907f5', 'rows': 1, 'cols': ['여성대상'], 'hint': "FROM 의 표 이름만 바꾸고, WHERE 의 값 'FEMALE' 은 대문자 그대로 두었는지 확인하세요.", 'ordered': False, 'optional': False}, 'Q5': {'sig': 'b2185a0b2644', 'osig': '3255743aed7f', 'rows': 5, 'cols': ['nct_id', 'enrollment'], 'hint': '많은 순은 DESC 입니다. 완료 조건(WHERE)은 지우지 말고 자리만 옮기세요.', 'ordered': True, 'optional': False}, 'Q6': {'sig': '90c57c427a98', 'osig': '90c57c427a98', 'rows': 1, 'cols': ['시험수'], 'hint': "3상 '또는' 4상이므로 OR 입니다 (AND 는 두 조건을 모두 만족). 두 값이 모두 조건에 남아 있는지 확인하세요.", 'ordered': False, 'optional': False}, 'Q7': {'sig': 'b27f1795ebc1', 'osig': 'c71c44d62a60', 'rows': 9, 'cols': ['overall_status', '시험수'], 'hint': 'SELECT 에 적은 일반 열(집계 함수가 아닌 열)은 GROUP BY 에도 적습니다. 오류 메시지가 권하는 ANY_VALUE 는 쓰지 않습니다.', 'ordered': False, 'optional': False}, 'Q8': {'sig': '1a81cbec9ec8', 'osig': '7522b6e2fd99', 'rows': 11, 'cols': ['국가', '시험수'], 'hint': '묶기 전 조건은 WHERE, 묶은 뒤(COUNT) 조건은 HAVING 입니다. removed = false 는 WHERE 에 그대로 둡니다.', 'ordered': False, 'optional': False}, 'Q9': {'sig': '526b49a01f30', 'osig': '526b49a01f30', 'rows': 1, 'cols': ['평균인원'], 'hint': 'AVG 안의 열 이름만 바꾸세요. ROUND(…, 1) 과 관찰 연구 조건(WHERE)은 그대로 둡니다.', 'ordered': False, 'optional': False}, 'Q10': {'by_mesh': {'Lung Neoplasms': {'sig': '3933143b3674', 'osig': '3933143b3674', 'rows': 1, 'cols': ['완료시험']}, 'Parkinson Disease': {'sig': '91e783c88bbd', 'osig': '91e783c88bbd', 'rows': 1, 'cols': ['완료시험']}, 'Atrial Fibrillation': {'sig': 'ec3a9923c5ab', 'osig': 'ec3a9923c5ab', 'rows': 1, 'cols': ['완료시험']}, 'Breast Neoplasms': {'sig': 'f9cd9916ca85', 'osig': 'f9cd9916ca85', 'rows': 1, 'cols': ['완료시험']}, 'Depressive Disorder': {'sig': 'a1a23b29b049', 'osig': 'a1a23b29b049', 'rows': 1, 'cols': ['완료시험']}, 'Coronary Artery Disease': {'sig': '6b2956c6d841', 'osig': '6b2956c6d841', 'rows': 1, 'cols': ['완료시험']}, 'Heart Failure': {'sig': '19789e569dde', 'osig': '19789e569dde', 'rows': 1, 'cols': ['완료시험']}, 'Pulmonary Disease, Chronic Obstructive': {'sig': 'de3680ea15b9', 'osig': 'de3680ea15b9', 'rows': 1, 'cols': ['완료시험']}, 'Renal Insufficiency, Chronic': {'sig': '12154912c791', 'osig': '12154912c791', 'rows': 1, 'cols': ['완료시험']}, 'Alzheimer Disease': {'sig': 'f9b400466326', 'osig': 'f9b400466326', 'rows': 1, 'cols': ['완료시험']}, 'Obesity': {'sig': '2b784cd361ba', 'osig': '2b784cd361ba', 'rows': 1, 'cols': ['완료시험']}, 'COVID-19': {'sig': 'dec9985a1f5b', 'osig': 'dec9985a1f5b', 'rows': 1, 'cols': ['완료시험']}, 'Osteoarthritis': {'sig': '5c8ac86fa7ab', 'osig': '5c8ac86fa7ab', 'rows': 1, 'cols': ['완료시험']}, 'Sepsis': {'sig': '10afd058cebb', 'osig': '10afd058cebb', 'rows': 1, 'cols': ['완료시험']}, 'Pneumonia': {'sig': '61cea4e4bb26', 'osig': '61cea4e4bb26', 'rows': 1, 'cols': ['완료시험']}}, 'hint': "질환 조건과 완료 조건(WHERE · AND 두 줄)은 그대로 두었는지, 'COMPLETED' 를 대문자로 적었는지 확인하세요.", 'ordered': False, 'optional': False}, 'Q11': {'by_mesh': {'Lung Neoplasms': {'sig': '9c17dad798c4', 'osig': '7a8bd25c5fa3', 'rows': 8, 'cols': ['상태', '시험수']}, 'Parkinson Disease': {'sig': '66bac5b00c8d', 'osig': 'cb7eabfd56eb', 'rows': 9, 'cols': ['상태', '시험수']}, 'Atrial Fibrillation': {'sig': '4d597a3292de', 'osig': 'c039d7f9a3d1', 'rows': 8, 'cols': ['상태', '시험수']}, 'Breast Neoplasms': {'sig': '8c220a50f458', 'osig': 'a389ee624eb9', 'rows': 9, 'cols': ['상태', '시험수']}, 'Depressive Disorder': {'sig': 'a02ffbdacf27', 'osig': '1049b0afa191', 'rows': 9, 'cols': ['상태', '시험수']}, 'Coronary Artery Disease': {'sig': '2b38cb1fbfca', 'osig': '7f48bdecfe70', 'rows': 8, 'cols': ['상태', '시험수']}, 'Heart Failure': {'sig': '91180b134405', 'osig': '5be26bfb6548', 'rows': 9, 'cols': ['상태', '시험수']}, 'Pulmonary Disease, Chronic Obstructive': {'sig': 'a6a328db1457', 'osig': '02a534265645', 'rows': 9, 'cols': ['상태', '시험수']}, 'Renal Insufficiency, Chronic': {'sig': 'b5332180aa82', 'osig': 'f0b08c1f1f97', 'rows': 9, 'cols': ['상태', '시험수']}, 'Alzheimer Disease': {'sig': '14cd724ecc86', 'osig': '1bf458971524', 'rows': 9, 'cols': ['상태', '시험수']}, 'Obesity': {'sig': 'c205ec8bc172', 'osig': 'cfae1a03e218', 'rows': 9, 'cols': ['상태', '시험수']}, 'COVID-19': {'sig': '8a85648d4a9f', 'osig': 'ae3d49cd27d3', 'rows': 9, 'cols': ['상태', '시험수']}, 'Osteoarthritis': {'sig': '0057e02278cf', 'osig': 'f39d8bb03f89', 'rows': 9, 'cols': ['상태', '시험수']}, 'Sepsis': {'sig': 'b892ad1abe1d', 'osig': 'ac40f16d79bf', 'rows': 9, 'cols': ['상태', '시험수']}, 'Pneumonia': {'sig': '2f01833da2e5', 'osig': '69da58f6d846', 'rows': 9, 'cols': ['상태', '시험수']}}, 'hint': 'ON 에는 두 표를 맞출 열(양쪽 표의 nct_id)을 적습니다. 질환 조건은 WHERE 에 그대로 둡니다.', 'ordered': False, 'optional': False}, 'Q12': {'by_mesh': {'Lung Neoplasms': {'sig': 'a1e2fe75b9ca', 'osig': 'b192a65e8df5', 'rows': 9, 'cols': ['단계', '시험수']}, 'Parkinson Disease': {'sig': '4c52781343b2', 'osig': 'ba34be9b3b23', 'rows': 9, 'cols': ['단계', '시험수']}, 'Atrial Fibrillation': {'sig': '60b236dbfe89', 'osig': 'ac4cada0f0ea', 'rows': 8, 'cols': ['단계', '시험수']}, 'Breast Neoplasms': {'sig': 'fddea3980c3c', 'osig': '34e752f01e26', 'rows': 9, 'cols': ['단계', '시험수']}, 'Depressive Disorder': {'sig': 'e46264b334d9', 'osig': '44d2b1538065', 'rows': 9, 'cols': ['단계', '시험수']}, 'Coronary Artery Disease': {'sig': 'f3cbf948af16', 'osig': '66c8f772dd8c', 'rows': 8, 'cols': ['단계', '시험수']}, 'Heart Failure': {'sig': 'dc0c6f1724d3', 'osig': '854a87803a4e', 'rows': 9, 'cols': ['단계', '시험수']}, 'Pulmonary Disease, Chronic Obstructive': {'sig': 'a841485998e8', 'osig': '38f09ae8cd4f', 'rows': 9, 'cols': ['단계', '시험수']}, 'Renal Insufficiency, Chronic': {'sig': '9219f66e9820', 'osig': '8f118f5c9532', 'rows': 9, 'cols': ['단계', '시험수']}, 'Alzheimer Disease': {'sig': '5a32e9fb8185', 'osig': '105fe5d27872', 'rows': 9, 'cols': ['단계', '시험수']}, 'Obesity': {'sig': '4e74d31e2f4f', 'osig': '382f59729a22', 'rows': 9, 'cols': ['단계', '시험수']}, 'COVID-19': {'sig': '5029bb18f18b', 'osig': 'a80e9a74cab1', 'rows': 9, 'cols': ['단계', '시험수']}, 'Osteoarthritis': {'sig': 'eadeef65d0f6', 'osig': 'ffca135b7f71', 'rows': 9, 'cols': ['단계', '시험수']}, 'Sepsis': {'sig': '9f86ecc30cbd', 'osig': '140b0c06c8d8', 'rows': 9, 'cols': ['단계', '시험수']}, 'Pneumonia': {'sig': 'cbd822d5cdf7', 'osig': '94351564df12', 'rows': 9, 'cols': ['단계', '시험수']}}, 'hint': 'ON 의 양쪽은 서로 다른 두 표(s 와 b)의 nct_id 여야 합니다. 질환 조건은 그대로 둡니다.', 'ordered': False, 'optional': False}}

## 데이터 구성 — 어떤 표가 있고, 어떻게 연결되나

문제를 풀기 전에 표들의 관계를 먼저 봅니다. 열이 어느 표에 있는지, 표를 어떻게 연결하는지 막히면 이 그림으로 돌아오세요.

![aact_mini 데이터 구성 (ERD)](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | studies 와의 관계 | 주로 쓰는 열 |
|---|---|---|---|---|---|
| `studies` | 임상시험 1건 | `nct_id` | — | 기준이 되는 표 | brief_title, study_type, overall_status, phase, start_date, enrollment (숫자), enrollment_type (ACTUAL · ESTIMATED 같은 글자), results_first_posted_date |
| `eligibilities` | 선정 · 제외 기준 | `id` | `nct_id` → studies | 1 : 1 (시험마다 1행) | gender, minimum_age (글자) |
| `sponsors` | 스폰서 1곳 | `id` | `nct_id` → studies | 1 : N (하나 이상) | agency_class, lead_or_collaborator, name |
| `conditions` | 대상 질환 1개 (연구자가 적은 이름) | `id` | `nct_id` → studies | 1 : N (하나 이상) | name (자유 입력이라 표기가 제각각) |
| `browse_conditions` | 표준 질환명(MeSH) 1개 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | mesh_term |
| `countries` | 참여 국가 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, removed (TRUE 면 등록 후 빠진 국가) |
| `facilities` | 수행 기관 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, country |

**읽는 법** (5 · 6주차)

1. **기본키** — 한 행을 하나로 찾는 열입니다. `studies` 는 `nct_id`(임상시험 등록번호), 나머지 표는 `id` 입니다.
2. **외래키** — 다른 표의 `nct_id` 는 `studies.nct_id` 를 가리킵니다. 그래서 표를 연결(JOIN)할 때는 늘 `ON s.nct_id = 다른표.nct_id` 로 잇습니다.
3. **1 : N** — 시험 1건에 국가 · 기관 · 질환이 여러 개면 그 표에는 행이 여러 개입니다. 연결하면 시험 1건이 여러 행이 되므로, 시험 수는 `COUNT(DISTINCT s.nct_id)` 로 셉니다. 그림에서 `O` 표시가 있는 표는 행이 하나도 없는 시험도 있어서, INNER JOIN 하면 그 시험이 빠집니다.

In [ ]:
%%sql
-- 확인: studies 에서는 1행인 시험이 countries 와 연결하면 국가 수만큼 여러 행이 됩니다
SELECT s.nct_id, s.overall_status, c.name AS 국가
FROM studies AS s
JOIN countries AS c ON s.nct_id = c.nct_id
WHERE s.nct_id = 'NCT01795521'

## 1. 조회 기초 (7주차) — Q1~Q6

적는 순서: SELECT → FROM → WHERE → ORDER BY → LIMIT (슬라이드 174). 문자 · 날짜는 '작은따옴표', 숫자는 따옴표 없이 (슬라이드 179).

**Q1.** 동료가 원한 것: 지금 모집 중(`RECRUITING`)인 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q1
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 모집중
FROM studies
WHERE overall_status = RECRUITING;

In [ ]:
check("Q1", _)

✏️ **Q1 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q2.** 동료가 원한 것: 2023년 1월 1일부터(그날 포함) 시작한 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q2
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE start_date >= 2023;

In [ ]:
check("Q2", _)

✏️ **Q2 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q3.** 동료가 원한 것: 등록 인원이 1000명 이상인 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q3
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE enrollment >= '1000명';

In [ ]:
check("Q3", _)

✏️ **Q3 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q4.** 동료가 원한 것: 여성만 참여할 수 있는 시험(`gender` 가 `FEMALE`)이 몇 건인지 세어 주세요. (열이 어느 표에 있는지는 위 **데이터 구성** 표에서 확인)

In [ ]:
%%sql
-- Q4
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 여성대상
FROM studies
WHERE gender = 'FEMALE';

In [ ]:
check("Q4", _)

✏️ **Q4 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q5.** 동료가 원한 것: 완료(`COMPLETED`)된 시험 중 등록 인원이 가장 많은 5건을 많은 순서대로 보여 주세요.

In [ ]:
%%sql
-- Q5
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT nct_id, enrollment
FROM studies
ORDER BY enrollment DESC
WHERE overall_status = 'COMPLETED'
LIMIT 5;

In [ ]:
check("Q5", _)

✏️ **Q5 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q6.** 동료가 원한 것: 3상 또는 4상(`PHASE3`, `PHASE4`) 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q6
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE phase = 'PHASE3' OR 'PHASE4';

In [ ]:
check("Q6", _)

✏️ **Q6 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 2. 집계와 GROUP BY (8주차) — Q7~Q9

적는 순서: SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT (슬라이드 221). WHERE 는 묶기 전, HAVING 은 묶은 뒤 조건입니다.

**Q7.** 동료가 원한 것: 진행 상태(`overall_status`)별로 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q7
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT overall_status, COUNT(*) AS 시험수
FROM studies;

In [ ]:
check("Q7", _)

✏️ **Q7 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q8.** 동료가 원한 것: 참여 시험이 1000건 이상인 국가만, 국가별 시험 수를 많은 순으로 보여 주세요. (제외된 국가 행은 빼고)

In [ ]:
%%sql
-- Q8
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT name AS 국가, COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
  AND COUNT(DISTINCT nct_id) >= 1000
GROUP BY name
ORDER BY 시험수 DESC;

In [ ]:
check("Q8", _)

✏️ **Q8 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q9.** 동료가 원한 것: 관찰 연구(`OBSERVATIONAL`)의 평균 등록 인원(명)을 소수 첫째 자리까지 보여 주세요.

In [ ]:
%%sql
-- Q9
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT ROUND(AVG(enrollment_type), 1) AS 평균인원
FROM studies
WHERE study_type = 'OBSERVATIONAL';

In [ ]:
check("Q9", _)

✏️ **Q9 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 3. 배정 질환 분석 (JOIN) — Q10~Q12

9주차 종합 예제(슬라이드 248 · 249)와 같은 모양입니다. 두 표를 잇는 조건은 ON 절에 적습니다 (슬라이드 225). 준비 셀에 나온 **내 배정 질환**이 `'{{MY_MESH}}'` 자리에 자동으로 들어갑니다. `'{{MY_MESH}}'` 는 지우거나 바꾸지 마세요.

**Q10.** 동료가 원한 것: 배정 질환 시험 중 완료(`COMPLETED`)된 시험이 몇 건인지 세어 주세요.

In [ ]:
%%sql
-- Q10
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT nct_id) AS 완료시험
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = '{{MY_MESH}}'
  AND s.overall_status = 'COMPLETED';

In [ ]:
check("Q10", _)

✏️ **Q10 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q11.** 동료가 원한 것: 배정 질환 시험을 진행 상태별로 나누어 시험 수를 많은 순으로 보여 주세요.

In [ ]:
%%sql
-- Q11
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT s.overall_status AS 상태, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
WHERE b.mesh_term = '{{MY_MESH}}'
GROUP BY s.overall_status
ORDER BY 시험수 DESC;

In [ ]:
check("Q11", _)

✏️ **Q11 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q12.** 동료가 원한 것: 배정 질환 시험을 임상 단계(`phase`)별로 나누어 시험 수를 많은 순으로 보여 주세요.

In [ ]:
%%sql
-- Q12
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT s.phase AS 단계, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON studies.nct_id = b.nct_id
WHERE b.mesh_term = '{{MY_MESH}}'
GROUP BY s.phase
ORDER BY 시험수 DESC;

In [ ]:
check("Q12", _)

✏️ **Q12 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 4. 결과 해석 (5점)

자동 채점하지 않습니다. 위 Q10~Q12 의 **내 결과 숫자**를 넣어 문장으로 쓰세요.

**해석1 (3점) — 배정 질환의 임상시험 현황.** 아래 세 가지를 2~3문장으로 쓰세요.

1. 배정 질환 시험은 모두 몇 건인가요? (Q11 의 시험수를 모두 더한 값)
2. 그중 완료된 시험(Q10)은 몇 건이고, 몇 % 인가요? (Q10 ÷ (1번 값) × 100, 계산기를 써도 됩니다, 소수 첫째 자리까지)
3. Q11 에서 지금 모집 중(`RECRUITING`)인 시험은 몇 건인가요? 이 숫자는 환자 수인가요, 시험 수인가요? (슬라이드 242 ② 세는 단위)

✏️ **해석1** (두 번 눌러 2~3문장으로):
→ 

**해석2 (2점) — 검산.** Q12 의 시험수를 모두 더하세요. 단계가 `NA` 인 행과 `None`(빈 값)인 행도 시험이므로 모두 더합니다 (둘은 다른 값, 슬라이드 211). 이 합계를 해석1 의 1번 값과 비교해, 두 합계가 같은지와 같아야 하는 이유를 한두 문장으로 쓰세요. (슬라이드 242 ② 세는 단위 · 248 검산)

✏️ **해석2** (두 번 눌러 한두 문장으로):
→ 

## 5. 제출 전 확인

- [ ] 맨 위 셀에 학번과 이름을 입력했다
- [ ] Q1~Q12 가 모두 ✅ 이다 (못 고친 문항은 그 문항만 0점)
- [ ] 문항마다 ‘무엇이 왜 틀렸나’ 를 한 줄씩 적었다 (12개)
- [ ] 해석1 · 해석2 를 문장으로 썼다
- [ ] 메뉴 `런타임 → 모두 실행` 후 아래 셀로 결과를 확인하고 저장했다. 못 고친 문항에서 멈추면 그 아래 셀을 누르고 `런타임 → 이후 셀 실행` 을 누릅니다.

In [ ]:
제출확인()